# BirdCLEF 2026 — Perch Embedding Precompute v3 (Kaggle)
## Computes Perch ONNX embeddings for train audio + train soundscapes
## Output saved to `/kaggle/working/perch_embeddings_v3/` — save as Output Dataset

### Required Kaggle inputs
1. **Competition data** — `birdclef-2026` (added automatically as competition)
2. **Perch ONNX** — add a public dataset containing `perch_v2_cpu.onnx`:
   - Search Kaggle for **`perch onnx birdclef 2026`** and add whichever matches
   - The notebook auto-detects the file under `/kaggle/input/`

### After this notebook finishes
- Go to **Output** tab → **New Dataset** → name it `birdclef-2026-perch-embs-v3`
- Add that dataset as input to `birdclef2026-train-v23-kaggle.ipynb`

In [ ]:
# === CELL 1: INSTALL & IMPORTS ===
import subprocess, sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'onnxruntime', 'soundfile', 'librosa', 'tqdm'],
    check=False,
)

import os, gc, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import onnxruntime as ort
from tqdm import tqdm

PERCH_SR      = 32000
SECONDS       = 5
PERCH_TARGET  = PERCH_SR * SECONDS   # 160,000 samples
_PERCH_BATCH  = 16                   # smaller batch for Kaggle CPU RAM
_MODEL_LABEL  = 'perch_v2_cpu_onnx'

print(f'onnxruntime {ort.__version__}')
print(f'Providers: {ort.get_available_providers()}')

In [ ]:
# === CELL 2: PATHS ===
# Competition data
COMP_DIR              = '/kaggle/input/birdclef-2026'
TRAIN_META_CSV        = f'{COMP_DIR}/train.csv'
TRAIN_AUDIO_DIR       = f'{COMP_DIR}/train_audio'
SOUNDSCAPE_DIR        = f'{COMP_DIR}/train_soundscapes'
SOUNDSCAPE_LABELS_CSV = f'{COMP_DIR}/train_soundscapes_labels.csv'

# Output (Kaggle saves /kaggle/working as output dataset)
EMBD_DIR = Path('/kaggle/working/perch_embeddings_v3')
EMBD_DIR.mkdir(parents=True, exist_ok=True)

# ONNX model — auto-detect from all .onnx files under /kaggle/input
_onnx_candidates = [
    # known slugs — add whichever public dataset you found
    '/kaggle/input/birdclef-2026-perch-onnx/perch_v2_cpu.onnx',
    '/kaggle/input/perch-onnx-for-birdclef-2026/perch_v2_cpu.onnx',
    '/kaggle/input/perch-onnx-birdclef-2026/perch_v2_cpu.onnx',
    '/kaggle/input/perch-onnx-birdclef2026/perch_v2_cpu.onnx',
]
# Also glob for any .onnx under /kaggle/input in case the slug differs
for _f in Path('/kaggle/input').glob('**/*.onnx'):
    _onnx_candidates.append(str(_f))

ONNX_PATH = None
for _c in _onnx_candidates:
    if Path(_c).exists():
        ONNX_PATH = _c
        print(f'Found ONNX: {ONNX_PATH}')
        break

if ONNX_PATH is None:
    raise RuntimeError(
        'perch_v2_cpu.onnx not found under /kaggle/input.\n'
        'Add the Perch ONNX dataset to this notebook via the Data tab.\n'
        'Search Kaggle for: perch onnx birdclef 2026'
    )

df    = pd.read_csv(TRAIN_META_CSV)
sc_df = pd.read_csv(SOUNDSCAPE_LABELS_CSV)

print(f'Training clips        : {len(df)}')
print(f'Soundscape rows       : {len(sc_df)}')
print(f'TRAIN_AUDIO_DIR exists: {Path(TRAIN_AUDIO_DIR).exists()}')
print(f'SOUNDSCAPE_DIR exists : {Path(SOUNDSCAPE_DIR).exists()}')
print(f'ONNX_PATH             : {ONNX_PATH}')
print(f'EMBD_DIR              : {EMBD_DIR}')
print(f'Already cached        : {len(list(EMBD_DIR.glob("*.npy")))} embeddings')

In [ ]:
# === CELL 3: LOAD ONNX SESSION + SANITY CHECK ===
print(f'Loading ONNX session from {ONNX_PATH} ...')

_sess_opts = ort.SessionOptions()
_sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
_sess_opts.intra_op_num_threads = os.cpu_count() or 4
_ort_session = ort.InferenceSession(
    ONNX_PATH,
    sess_options=_sess_opts,
    providers=['CPUExecutionProvider'],
)
_onnx_inp = _ort_session.get_inputs()[0].name

# Auto-detect 1536-d embedding output
_onnx_emb_key = None
for _out in _ort_session.get_outputs():
    if _out.shape and _out.shape[-1] == 1536:
        _onnx_emb_key = _out.name
        break

if _onnx_emb_key is None:
    _dummy = np.zeros((1, PERCH_TARGET), dtype=np.float32)
    _dummy_outs = _ort_session.run(None, {_onnx_inp: _dummy})
    for _o, _ov in zip(_ort_session.get_outputs(), _dummy_outs):
        if _ov.ndim >= 2 and _ov.shape[-1] == 1536:
            _onnx_emb_key = _o.name
            break
assert _onnx_emb_key is not None, 'Cannot find 1536-d output in ONNX model.'

_out_names = [o.name for o in _ort_session.get_outputs()]
_emb_idx   = _out_names.index(_onnx_emb_key)

# Load model_info.json if present (written by perch-to-onnx notebook)
_info_json = Path(ONNX_PATH).parent / 'model_info.json'
if _info_json.exists():
    _info = json.load(open(_info_json))
    _onnx_inp     = _info.get('input_name', _onnx_inp)
    _onnx_emb_key = _info.get('embedding_output_name', _onnx_emb_key)
    _emb_idx      = _out_names.index(_onnx_emb_key)
    print(f'Loaded model_info.json: input={_onnx_inp!r}, emb={_onnx_emb_key!r}')

# Sanity check
_test_audio = np.sin(2 * np.pi * 440 * np.linspace(0, 5, PERCH_TARGET)).astype(np.float32)[None]
_test_outs  = _ort_session.run(None, {_onnx_inp: _test_audio})
_test_emb   = _test_outs[_emb_idx]
if _test_emb.ndim == 3:
    _test_emb = _test_emb.mean(axis=1)
_test_emb = _test_emb[0]
assert _test_emb.shape == (1536,), f'Expected (1536,) but got {_test_emb.shape}'
assert _test_emb.std() > 0.1, f'Embedding std={_test_emb.std():.4f} near zero — ONNX may be corrupt'
print(f'✅ Sanity check passed: shape={_test_emb.shape}, mean={_test_emb.mean():.4f}, std={_test_emb.std():.4f}')
del _test_audio, _test_outs, _test_emb

In [ ]:
# === CELL 4: FOCAL CLIP EMBEDDINGS ===
assert Path(TRAIN_AUDIO_DIR).exists(), f'Training audio not found at {TRAIN_AUDIO_DIR}'

print('Extracting focal clip embeddings ...')

_audio_batch, _name_batch = [], []

def _flush_batch():
    batch = np.stack(_audio_batch)  # (B, 160000)
    outs  = _ort_session.run(None, {_onnx_inp: batch})
    embs  = outs[_emb_idx]
    if embs.ndim == 3:
        embs = embs.mean(axis=1)
    for name, emb in zip(_name_batch, embs):
        np.save(str(EMBD_DIR / name), emb.astype(np.float32))
    _audio_batch.clear()
    _name_batch.clear()

failed  = 0
skipped = 0

for _, row in tqdm(df.iterrows(), total=len(df), desc='Focal embed'):
    audio_path = Path(TRAIN_AUDIO_DIR) / row['filename']
    emb_name   = row['filename'].replace('/', '_') + '.npy'
    emb_path   = EMBD_DIR / emb_name

    if emb_path.exists():
        skipped += 1
        continue

    try:
        _fi             = sf.info(str(audio_path))
        _frames_to_read = int(PERCH_TARGET * (_fi.samplerate / PERCH_SR)) + 4096
        y, sr0          = sf.read(str(audio_path), frames=_frames_to_read, always_2d=False)
        if y.ndim == 2:
            y = y.mean(axis=1)
        if sr0 != PERCH_SR:
            y = librosa.resample(y.astype(np.float32), orig_sr=sr0, target_sr=PERCH_SR)
        y = y.astype(np.float32)
        if len(y) < PERCH_TARGET:
            y = np.pad(y, (0, PERCH_TARGET - len(y)))
        else:
            y = y[:PERCH_TARGET]
        _audio_batch.append(y)
        _name_batch.append(emb_name)
        if len(_audio_batch) >= _PERCH_BATCH:
            _flush_batch()
    except Exception as e:
        failed += 1
        if failed <= 5:
            print(f'  WARN focal failed: {row["filename"]} — {e}')

if _audio_batch:
    _flush_batch()

focal_saved = len([f for f in EMBD_DIR.glob('*.npy') if not f.stem.startswith('soundscape_')])
print(f'\nFocal done! saved={focal_saved}  skipped={skipped}  failed={failed}')

In [ ]:
# === CELL 5: SOUNDSCAPE WINDOW EMBEDDINGS ===
assert Path(SOUNDSCAPE_DIR).exists(), f'Soundscape dir not found: {SOUNDSCAPE_DIR}'

def _parse_hms_to_secs(s: str) -> int:
    parts = str(s).strip().split(':')
    return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])

_sc_wave_cache = {}
_sc_audio_batch, _sc_name_batch = [], []

def _flush_sc_batch():
    if not _sc_audio_batch:
        return
    batch = np.stack(_sc_audio_batch)
    outs  = _ort_session.run(None, {_onnx_inp: batch})
    embs  = outs[_emb_idx]
    if embs.ndim == 3:
        embs = embs.mean(axis=1)
    for name, emb in zip(_sc_name_batch, embs):
        np.save(str(EMBD_DIR / name), emb.astype(np.float32))
    _sc_audio_batch.clear()
    _sc_name_batch.clear()

failed_sc  = 0
skipped_sc = 0
sc_df_sorted = sc_df.sort_values('filename').reset_index(drop=True)

for _, row in tqdm(sc_df_sorted.iterrows(), total=len(sc_df_sorted), desc='Soundscape embed'):
    stem     = Path(str(row['filename'])).stem
    end_secs = _parse_hms_to_secs(row['end'])
    emb_name = f'soundscape_{stem}_{end_secs}s.npy'
    emb_path = EMBD_DIR / emb_name

    if emb_path.exists():
        skipped_sc += 1
        continue

    try:
        if stem not in _sc_wave_cache:
            audio_path = None
            for _ext in ['.ogg', '.wav', '.flac']:
                _cand = Path(SOUNDSCAPE_DIR) / f'{stem}{_ext}'
                if _cand.exists():
                    audio_path = _cand
                    break
            if audio_path is None:
                if failed_sc <= 3:
                    print(f'  WARN: soundscape not found: {stem}')
                failed_sc += 1
                continue
            y_full, sr0 = sf.read(str(audio_path), always_2d=False)
            if y_full.ndim == 2:
                y_full = y_full.mean(axis=1)
            if sr0 != PERCH_SR:
                y_full = librosa.resample(y_full.astype(np.float32), orig_sr=sr0, target_sr=PERCH_SR)
            _sc_wave_cache[stem] = y_full.astype(np.float32)
            if len(_sc_wave_cache) > 5:
                oldest = next(iter(_sc_wave_cache))
                del _sc_wave_cache[oldest]

        y_full     = _sc_wave_cache[stem]
        end_samp   = int(end_secs * PERCH_SR)
        start_samp = max(0, end_samp - PERCH_TARGET)
        clip       = y_full[start_samp:end_samp]
        if len(clip) < PERCH_TARGET:
            clip = np.pad(clip, (0, PERCH_TARGET - len(clip)))

        _sc_audio_batch.append(clip)
        _sc_name_batch.append(emb_name)
        if len(_sc_audio_batch) >= _PERCH_BATCH:
            _flush_sc_batch()

    except Exception as e:
        failed_sc += 1
        if failed_sc <= 5:
            print(f'  WARN soundscape failed: {stem} @ {end_secs}s — {e}')

if _sc_audio_batch:
    _flush_sc_batch()

del _sc_wave_cache
gc.collect()

soundscape_saved = len([f for f in EMBD_DIR.glob('soundscape_*.npy')])
print(f'\nSoundscape done! saved={soundscape_saved}  skipped={skipped_sc}  failed={failed_sc}')
del _ort_session
gc.collect()
print('ONNX session released.')

In [ ]:
# === CELL 6: VERIFY & NEXT STEPS ===
all_emb_files = list(EMBD_DIR.glob('*.npy'))
focal_files   = [f for f in all_emb_files if not f.stem.startswith('soundscape_')]
sc_files      = [f for f in all_emb_files if f.stem.startswith('soundscape_')]

print(f'Total embeddings    : {len(all_emb_files)}')
print(f'  Focal embeds      : {len(focal_files)}')
print(f'  Soundscape embeds : {len(sc_files)}')

for _sample_f in random.sample(all_emb_files, min(5, len(all_emb_files))):
    _emb = np.load(str(_sample_f))
    assert _emb.shape == (1536,), f'Bad shape {_emb.shape} in {_sample_f.name}'
    assert _emb.std() > 0.05, f'Near-zero embedding in {_sample_f.name} (std={_emb.std():.4f})'

print(f'✅ Shape+std check passed')
print(f'   Model used: {_MODEL_LABEL}')
if not sc_files:
    print('⚠️  WARNING: no soundscape embeddings — Cell 5 may have failed')

print()
print('Next steps:')
print('  1. Output tab -> New Dataset -> name: birdclef-2026-perch-embs-v3')
print('  2. Add that dataset as input to birdclef2026-train-v23-kaggle.ipynb')